# Study 884 — Convexity Barbell — the teardown

The duration ladder & match, the spread's Newey-West *t* and bootstrap CI, the convexity regression + smile, the 2022 tell, the two-era cut, the leg-permutation placebo, the costed timer, and the 20-seed synthetic control.

In [1]:
R = {'start': '2010-01-04', 'end': '2026-06-30', 'n_bonds': 3, 'n_days': 3894, 'rows': 4147, 'fingerprint': '32356eb6aefe', 'beta_shy': 0.116, 'beta_ief': 0.864, 'beta_tlt': 2.02, 'w_short': 0.605, 'w_long': 0.395, 'barbell_ann': 2.15, 'barbell_vol': 6.41, 'barbell_mdd': -23.5, 'bullet_ann': 2.28, 'bullet_vol': 6.52, 'bullet_mdd': -23.9, 'corr': 0.944, 'spread_bps': -0.054, 't_nw': -0.27, 't_1s': -0.25, 'spread_sharpe': -0.06, 'boot_lo': -0.433, 'boot_hi': 0.344, 'sharpe_barbell_x': 0.147, 'sharpe_bullet_x': 0.165, 'sharpe_adv': -0.018, 'welch_t': -0.06, 'resid_dur_slope': 0.009, 'conv_slope': -0.222, 'spread_conv_bps': -0.047, 'spread_carry_bps': -0.007, 'smile': [0.305, 0.014, 0.053, -0.693, 0.052], 'y2021_bar': -1.46, 'y2021_bul': -3.33, 'y2021_sp': 1.89, 'y2022_bar': -15.4, 'y2022_bul': -15.16, 'y2022_sp': -0.38, 'y2025_bar': 4.79, 'y2025_bul': 8.03, 'y2025_sp': -3.06, 'era_early_bps': 0.034, 'era_early_t': 0.11, 'era_early_n': 1760, 'era_late_bps': -0.126, 'era_late_t': -0.47, 'era_late_n': 2134, 'placebo_obs': -0.054, 'placebo_mean': -0.129, 'placebo_sd': 0.061, 'placebo_p': 0.122, 'timer_05_net': -0.056, 'timer_1_net': -0.057, 'timer_2_net': -0.061, 'timer_1_t': -0.26, 'timer_1_ann': -0.14, 'turnover': 0.0037, 'null_mean_t': 0.01, 'null_sd_t': 0.73, 'null_fire': 0, 'planted_t': 4.3, 'planted_conv_slope': 0.7}

## The duration ladder & the match

Each bond's empirical duration = its trailing-252d beta to the equal-weight rates factor; solving `w·β_SHY + (1-w)·β_TLT = β_IEF` gives the barbell weight.

In [2]:
print('empirical durations: SHY %.3f  IEF %.3f  TLT %.3f'
      % (R['beta_shy'], R['beta_ief'], R['beta_tlt']))
print('=> barbell = %.3f*SHY + %.3f*TLT (duration-matched to IEF)'
      % (R['w_short'], R['w_long']))
print('spread residual duration slope on the factor = %+.4f (~0 => matched)'
      % R['resid_dur_slope'])

empirical durations: SHY 0.116  IEF 0.864  TLT 2.020
=> barbell = 0.605*SHY + 0.395*TLT (duration-matched to IEF)
spread residual duration slope on the factor = +0.0090 (~0 => matched)


## The headline — barbell vs bullet, total return

In [3]:
print(f"barbell : {R['barbell_ann']:+.2f}%/yr  vol {R['barbell_vol']:.2f}%  maxDD {R['barbell_mdd']:.1f}%")
print(f"bullet  : {R['bullet_ann']:+.2f}%/yr  vol {R['bullet_vol']:.2f}%  maxDD {R['bullet_mdd']:.1f}%  (corr {R['corr']:.3f})")
print(f"spread  : {R['spread_bps']:+.3f} bps/day  NW(10) t = {R['t_nw']:+.2f}  one-sample t = {R['t_1s']:+.2f}  Sharpe = {R['spread_sharpe']:+.3f}")
print(f"bootstrap spread-mean CI95 = [{R['boot_lo']:+.3f}, {R['boot_hi']:+.3f}] bps (straddles zero)")
print(f"excess-vs-excess Sharpe: barbell {R['sharpe_barbell_x']:+.3f} vs bullet {R['sharpe_bullet_x']:+.3f} -> advantage {R['sharpe_adv']:+.3f} (Welch t = {R['welch_t']:+.2f})")

barbell : +2.15%/yr  vol 6.41%  maxDD -23.5%
bullet  : +2.28%/yr  vol 6.52%  maxDD -23.9%  (corr 0.944)
spread  : -0.054 bps/day  NW(10) t = -0.27  one-sample t = -0.25  Sharpe = -0.060
bootstrap spread-mean CI95 = [-0.433, +0.344] bps (straddles zero)
excess-vs-excess Sharpe: barbell +0.147 vs bullet +0.165 -> advantage -0.018 (Welch t = -0.06)


## Convexity — the `f²` regression and the smile

Regress the spread on `[1, f, f²]`. The claim needs a **positive** `f²` slope (barbell captures its extra convexity); the tape gives a *wrong-signed* one, and the smile (mean spread by |move| quintile) shows no monotone rise into big moves.

In [4]:
print(f"convexity slope on f^2 = {R['conv_slope']:+.3f}  (claim: > 0 -> WRONG-SIGNED)")
print(f"mean-spread split: convexity {R['spread_conv_bps']:+.4f} + carry/drift {R['spread_carry_bps']:+.4f} bps")
labels=['small','.','.','.','big']
print('smile (mean spread bps by |move| quintile):')
for lab,v in zip(labels, R['smile']): print(f'   {lab:>5}: {v:+.3f}')

convexity slope on f^2 = -0.222  (claim: > 0 -> WRONG-SIGNED)
mean-spread split: convexity -0.0470 + carry/drift -0.0070 bps
smile (mean spread bps by |move| quintile):
   small: +0.305
       .: +0.014
       .: +0.053
       .: -0.693
     big: +0.052


## The 2022 tell — the biggest rate move in the sample

The claim says the barbell wins when yields move a lot. In 2022 — the historic selloff — it **lost** to the bullet.

In [5]:
for y,bar,bul,sp in [(2021,R['y2021_bar'],R['y2021_bul'],R['y2021_sp']),
                     (2022,R['y2022_bar'],R['y2022_bul'],R['y2022_sp']),
                     (2025,R['y2025_bar'],R['y2025_bul'],R['y2025_sp'])]:
    print(f'{y}: barbell {bar:+.2f}%  bullet {bul:+.2f}%  spread {sp:+.2f}%')

2021: barbell -1.46%  bullet -3.33%  spread +1.89%
2022: barbell -15.40%  bullet -15.16%  spread -0.38%
2025: barbell +4.79%  bullet +8.03%  spread -3.06%


## Robustness — two eras (split 2018-01-01)

In [6]:
print(f"2010-2017 (n={R['era_early_n']}): {R['era_early_bps']:+.3f} bps  NW t = {R['era_early_t']:+.2f}")
print(f"2018-2026 (n={R['era_late_n']}): {R['era_late_bps']:+.3f} bps  NW t = {R['era_late_t']:+.2f}")

2010-2017 (n=1760): +0.034 bps  NW t = +0.11
2018-2026 (n=2134): -0.126 bps  NW t = -0.47


## Placebo — permute the two barbell legs in time

Break the day-by-day alignment of the two legs; the observed spread should sit inside the placebo cloud (no convexity-alignment signal).

In [7]:
print(f"observed {R['placebo_obs']:+.3f} bps vs placebo mean {R['placebo_mean']:+.3f} "
      f"(sd {R['placebo_sd']:.3f}) -> right-tail p = {R['placebo_p']:.3f}")

observed -0.054 bps vs placebo mean -0.129 (sd 0.061) -> right-tail p = 0.122


## The timer — nothing to harvest, costs only subtract

The barbell turns over slowly, so frictions are tiny — but the gross spread is already ≈ 0, so the net is negative at every cost level.

In [8]:
for tag,net in [('0.5 bp',R['timer_05_net']),('1 bp',R['timer_1_net']),('2 bps',R['timer_2_net'])]:
    print(f"{tag:>6}: net {net:+.3f} bps/day")
print(f"turnover {R['turnover']:.4f}/day, net t = {R['timer_1_t']:+.2f}, ~{R['timer_1_ann']:+.2f}%/yr")

0.5 bp: net -0.056 bps/day
  1 bp: net -0.057 bps/day
 2 bps: net -0.061 bps/day
turnover 0.0037/day, net t = -0.26, ~-0.14%/yr


## Synthetic positive control — the machinery is unbiased

Live: the detector must NOT fire on the null (convexity fairly priced) and must recover a planted under-priced convexity — while the convexity slope is positive in *both* worlds (convexity is structural, an edge only when under-priced).

In [9]:
import os, sys
sys.path.insert(0, os.path.abspath('..'))
sys.path.insert(0, os.path.abspath(os.path.join('..','..','..')))
import numpy as np
from barbell import data, strategy as st
null_t = np.array([st.synthetic_detect(data.synthetic_panel(edge=0.0, seed=884+s, n_days=1300))['t_nw'] for s in range(8)])
print(f"null (edge=0), 8 seeds: spread NW t mean {null_t.mean():+.2f} (sd {null_t.std(ddof=1):.2f}), |t|>=2 in {(abs(null_t)>=2).sum()}/8")
planted = st.synthetic_detect(data.synthetic_panel(edge=0.6, seed=884, n_days=1800))
print(f"planted (edge=0.6): spread NW t = {planted['t_nw']:+.2f}, convexity slope = {planted['conv_slope']:+.3f}")

null (edge=0), 8 seeds: spread NW t mean -0.04 (sd 0.69), |t|>=2 in 0/8
planted (edge=0.6): spread NW t = +4.30, convexity slope = +0.697


## Verdict

- **Signal — None.** A duration-matched SHY+TLT barbell does **not** out-earn the IEF bullet: **+2.15%/yr vs +2.28%**, spread **-0.054 bps** (NW *t* = **-0.27**), bootstrap CI **[-0.43, +0.34]** straddling zero, Sharpe advantage **-0.02**, flat in both eras. The convexity is genuine but invisible in total return (`f²` slope -0.22, no smile, and 2022 went the wrong way). The 20-seed synthetic control recovers a *planted* edge cleanly (*t* = +4.3, 0/20 nulls fire), so the machinery is sound — the net edge is simply absent.
- **Tradability — Mirage.** No gross edge to cost; the spread is ≈ 0 before frictions and negative after (net ~-0.14%/yr). The free convexity lunch is fully offset by carry give-up and curve risk.